Types of Recommendation System:

- Content Based Recommendation System - Recommends based on users past intreset
- Popularity Based Recommendation System - Recommends based on popularity
- Collaborative Based Recommendation System - Recommends by grouping people based on watch patterns

# Importind Dependencies

In [1]:
import pandas as pd
import numpy as np
import difflib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Data Collection and Preprocessing

In [2]:
# Load the data

movie_data = pd.read_csv('movies.csv')

In [3]:
# First five rows

movie_data.head()

,index,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew,director
0,0,237000000,Action Adventure Fantasy Science Fiction,http://www.avatarmovie.com/,19995,culture clash future space war space colony so...,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,Sam Worthington Zoe Saldana Sigourney Weaver S...,"[{'name': 'Stephen E. Rivkin', 'gender': 0, 'd...",James Cameron
1,1,300000000,Adventure Fantasy Action,http://disney.go.com/disneypictures/pirates/,285,ocean drug abuse exotic island east india trad...,en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,Johnny Depp Orlando Bloom Keira Knightley Stel...,"[{'name': 'Dariusz Wolski', 'gender': 2, 'depa...",Gore Verbinski
2,2,245000000,Action Adventure Crime,http://www.sonypictures.com/movies/spectre/,206647,spy based on novel secret agent sequel mi6,en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,...,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,Daniel Craig Christoph Waltz L\u00e9a Seydoux ...,"[{'name': 'Thomas Newman', 'gender': 2, 'depar...",Sam Mendes
3,3,250000000,Action Crime Drama Thriller,http://www.thedarkknightrises.com/,49026,dc comics crime fighter terrorist secret ident...,en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,...,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,Christian Bale Michael Caine Gary Oldman Anne ...,"[{'name': 'Hans Zimmer', 'gender': 2, 'departm...",Christopher Nolan
4,4,260000000,Action Adventure Science Fiction,http://movies.disney.com/john-carter,49529,based on novel mars medallion space travel pri...,en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,...,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,Taylor Kitsch Lynn Collins Samantha Morton Wil...,"[{'name': 'Andrew Stanton', 'gender': 2, 'depa...",Andrew Stanton


In [4]:
# No of rows and cols

movie_data.shape

(4803, 24)

In [5]:
# Selecting the relevent Features for recommendation

selected_features = ['genres', 'keywords', 'tagline', 'cast', 'director']
print(selected_features)

['genres', 'keywords', 'tagline', 'cast', 'director']


In [6]:
# Checking Null values

movie_data.isnull().sum()

index                      0
budget                     0
genres                    28
homepage                3091
id                         0
keywords                 412
original_language          0
original_title             0
overview                   3
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    2
spoken_languages           0
status                     0
tagline                  844
title                      0
vote_average               0
vote_count                 0
cast                      43
crew                       0
director                  30
dtype: int64

In [7]:
# Replacing Null values with Null String

for features in selected_features:
    movie_data[features] = movie_data[features].fillna('')

In [8]:
movie_data.isnull().sum()

index                      0
budget                     0
genres                     0
homepage                3091
id                         0
keywords                   0
original_language          0
original_title             0
overview                   3
popularity                 0
production_companies       0
production_countries       0
release_date               1
revenue                    0
runtime                    2
spoken_languages           0
status                     0
tagline                    0
title                      0
vote_average               0
vote_count                 0
cast                       0
crew                       0
director                   0
dtype: int64

In [9]:
# Combining all 5 Features

combined_feature = movie_data['genres']+' '+movie_data['keywords']+' '+movie_data['tagline']+' '+movie_data['cast']+' '+movie_data['director']

In [10]:
combined_feature

0       Action Adventure Fantasy Science Fiction cultu...
1       Adventure Fantasy Action ocean drug abuse exot...
2       Action Adventure Crime spy based on novel secr...
3       Action Crime Drama Thriller dc comics crime fi...
4       Action Adventure Science Fiction based on nove...
                              ...                        
4798    Action Crime Thriller united states\u2013mexic...
4799    Comedy Romance  A newlywed couple's honeymoon ...
4800    Comedy Drama Romance TV Movie date love at fir...
4801      A New Yorker in Shanghai Daniel Henney Eliza...
4802    Documentary obsession camcorder crush dream gi...
Length: 4803, dtype: object

In [11]:
# Converting the text data to Feature Vectors

vectorizer = TfidfVectorizer()

In [12]:
feature_vectors = vectorizer.fit_transform(combined_feature)

In [13]:
print(feature_vectors)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 124266 stored elements and shape (4803, 17318)>
  Coords	Values
  (0, 201)	0.07860022416510505
  (0, 274)	0.09021200873707368
  (0, 5274)	0.11108562744414445
  (0, 13599)	0.1036413987316636
  (0, 5437)	0.1036413987316636
  (0, 3678)	0.21392179219912877
  (0, 3065)	0.22208377802661425
  (0, 5836)	0.1646750903586285
  (0, 14378)	0.33962752210959823
  (0, 16587)	0.12549432354918996
  (0, 3225)	0.24960162956997736
  (0, 14271)	0.21392179219912877
  (0, 4945)	0.24025852494110758
  (0, 15261)	0.07095833561276566
  (0, 16998)	0.1282126322850579
  (0, 11192)	0.09049319826481456
  (0, 11503)	0.27211310056983656
  (0, 13349)	0.15021264094167086
  (0, 17007)	0.23643326319898797
  (0, 17290)	0.20197912553916567
  (0, 13319)	0.2177470539412484
  (0, 14064)	0.20596090415084142
  (0, 16668)	0.19843263965100372
  (0, 14608)	0.15150672398763912
  (0, 8756)	0.22709015857011816
  :	:
  (4801, 403)	0.17727585190343229
  (4801, 4835)	0.247137650

# Cosine Similarity

In [14]:
similarity = cosine_similarity(feature_vectors)

In [15]:
print(similarity)

[[1.         0.07219487 0.037733   ... 0.         0.         0.        ]
 [0.07219487 1.         0.03281499 ... 0.03575545 0.         0.        ]
 [0.037733   0.03281499 1.         ... 0.         0.05389661 0.        ]
 ...
 [0.         0.03575545 0.         ... 1.         0.         0.02651502]
 [0.         0.         0.05389661 ... 0.         1.         0.        ]
 [0.         0.         0.         ... 0.02651502 0.         1.        ]]


In [16]:
similarity.shape

(4803, 4803)

# Getting Movie Name from User

In [17]:
movie_name = input("Enter Your Favourite Move Name: ")

In [18]:
print(movie_name)

dr strange


In [19]:
# Creating a list that contain all movies name in the given dataset

list_of_all_movies = movie_data['title'].tolist()

In [20]:
print(list_of_all_movies)

['Avatar', "Pirates of the Caribbean: At World's End", 'Spectre', 'The Dark Knight Rises', 'John Carter', 'Spider-Man 3', 'Tangled', 'Avengers: Age of Ultron', 'Harry Potter and the Half-Blood Prince', 'Batman v Superman: Dawn of Justice', 'Superman Returns', 'Quantum of Solace', "Pirates of the Caribbean: Dead Man's Chest", 'The Lone Ranger', 'Man of Steel', 'The Chronicles of Narnia: Prince Caspian', 'The Avengers', 'Pirates of the Caribbean: On Stranger Tides', 'Men in Black 3', 'The Hobbit: The Battle of the Five Armies', 'The Amazing Spider-Man', 'Robin Hood', 'The Hobbit: The Desolation of Smaug', 'The Golden Compass', 'King Kong', 'Titanic', 'Captain America: Civil War', 'Battleship', 'Jurassic World', 'Skyfall', 'Spider-Man 2', 'Iron Man 3', 'Alice in Wonderland', 'X-Men: The Last Stand', 'Monsters University', 'Transformers: Revenge of the Fallen', 'Transformers: Age of Extinction', 'Oz: The Great and Powerful', 'The Amazing Spider-Man 2', 'TRON: Legacy', 'Cars 2', 'Green Lant

In [21]:
# Finding the closest match for the movie name given by user

find_close_match = difflib.get_close_matches(movie_name, list_of_all_movies)

In [22]:
print(find_close_match)

['Tiger Orange', 'Arbitrage']


In [23]:
close_match = find_close_match[0]
print(close_match)

Tiger Orange


In [24]:
# Finding the index of movie with title

index_of_movie =  movie_data[movie_data.title == close_match]['index'].values[0]
print(index_of_movie)

4746


In [25]:
# Getting the list of similar movies

similarity_score = list(enumerate(similarity[index_of_movie]))
print(similarity_score)

[(0, np.float64(0.0)), (1, np.float64(0.0239303636279155)), (2, np.float64(0.0)), (3, np.float64(0.0032557685535941584)), (4, np.float64(0.02034460728497136)), (5, np.float64(0.0)), (6, np.float64(0.0)), (7, np.float64(0.017959077048344735)), (8, np.float64(0.0)), (9, np.float64(0.027198628874577017)), (10, np.float64(0.0)), (11, np.float64(0.0)), (12, np.float64(0.027995033745930677)), (13, np.float64(0.02527562641684086)), (14, np.float64(0.0)), (15, np.float64(0.0)), (16, np.float64(0.017029022601871262)), (17, np.float64(0.05371983658921549)), (18, np.float64(0.009980854423013427)), (19, np.float64(0.0)), (20, np.float64(0.0)), (21, np.float64(0.014932150810243077)), (22, np.float64(0.0)), (23, np.float64(0.0)), (24, np.float64(0.0027508368717724945)), (25, np.float64(0.0029358000127509324)), (26, np.float64(0.05614567595714239)), (27, np.float64(0.0)), (28, np.float64(0.0)), (29, np.float64(0.0)), (30, np.float64(0.010528552124156824)), (31, np.float64(0.0)), (32, np.float64(0.035

In [26]:
len(similarity_score)

4803

In [27]:
# Sorting the movie name based on similarity score

sorted_similarity_score = sorted(similarity_score, key=lambda x:x[1], reverse=True)
print(sorted_similarity_score)

[(4746, np.float64(0.9999999999999998)), (1376, np.float64(0.15084127426854338)), (2183, np.float64(0.13150689642002733)), (4255, np.float64(0.12123400658849395)), (2122, np.float64(0.11748373791149377)), (1865, np.float64(0.11719313103114087)), (4388, np.float64(0.11688462077914617)), (1556, np.float64(0.11476481852020226)), (502, np.float64(0.10801613259203459)), (723, np.float64(0.10311995589788506)), (733, np.float64(0.10291661752263286)), (231, np.float64(0.10083545018906355)), (2734, np.float64(0.10021417445678181)), (3538, np.float64(0.09749754680369965)), (52, np.float64(0.09737319716321922)), (1371, np.float64(0.09673700866555343)), (1275, np.float64(0.09652825701138393)), (2150, np.float64(0.09278300572595288)), (1207, np.float64(0.09275743756349339)), (618, np.float64(0.09187847040255884)), (955, np.float64(0.0916591867145132)), (4696, np.float64(0.09103955752271956)), (163, np.float64(0.09070380136598624)), (1152, np.float64(0.0897338188563295)), (3214, np.float64(0.0893314

In [28]:
# Print the name of similar movies based on index

print("Movie Suggested for you is : ")

i = 0

for movie in sorted_similarity_score:
    index = movie[0]
    title_from_index = movie_data[movie_data.index ==index]['title'].values[0]
    if (i<30):
        print(i, '.', title_from_index)
        i = i+1



Movie Suggested for you is : 
0 . Tiger Orange
1 . In & Out
2 . Home for the Holidays
3 . Growing Up Smith
4 . Epic Movie
5 . What a Girl Wants
6 . Hardflip
7 . Mystic River
8 . The Invasion
9 . The Happening
10 . Up Close & Personal
11 . Monsters, Inc.
12 . Miracles from Heaven
13 . Do the Right Thing
14 . Transformers: Dark of the Moon
15 . Trainwreck
16 . Sunshine
17 . Eye for an Eye
18 . Made of Honor
19 . Mystery Men
20 . The Peacemaker
21 . Weekend
22 . Watchmen
23 . Back to the Future Part II
24 . Barbarella
25 . 13 Hours: The Secret Soldiers of Benghazi
26 . The Thomas Crown Affair
27 . The Boys from Brazil
28 . All the King's Men
29 . The Lords of Salem


# Movie Recommendation System

In [29]:
movie_name = input("Enter Your Favourite Movie Name: ")
list_of_all_titles = movie_data['title'].tolist()
find_close_match = difflib.get_close_matches(movie_name, list_of_all_titles)
close_match = find_close_match[0]
index_of_movie = movie_data[movie_data.title == close_match]['index'].values[0]
similarity_score = list(enumerate(similarity[index_of_movie]))
sorted_similarity_score = sorted(similarity, key=lambda x:x[1], reverse=True)